# Multi-Ticker 200 SMA Breakout Accumulation vs Monthly DCA (With ADX Regime Filter)

**Strategy Overview (per ticker):**
- **Signal Strategy (Breakout Accumulation):**
  - Uses a selective 200 SMA breakout with an ADX(14) regime filter on the daily timeframe.
  - **Regime Filter:** ADX(14) > 25 on the breakout day (previous close); skip if ADX ≤ 20 (choppy/low trend).
  - **Breakout:** Close > SMA(200) and previous Close ≤ SMA(200) (first close above 200 SMA after being below/at it).
  - **Buy Timing:** Next trading day after the breakout; buy at Open if Open > SMA(200) and ADX.shift(1) > 25.
  - **Buy Size:** $10,000 per valid entry.
  - **Caps:** Maximum 12 buys per calendar year (per ticker). No global $120k cap across years.
  - **No stop loss, no exits:** All positions are held indefinitely (pure accumulation).
- **DCA Benchmark (Monthly, per ticker):**
  - Buy $10,000 at month-end closing price **each month** (12 buys per year), over the full period.
  - Total invested per ticker = number_of_months_in_period × $10,000.
- **Tickers:** QQQ, TSLA, MSFT, GOOG, AAPL
- **Period:** January 2011 to December 2025
- **Final Valuation:** At last available Adj Close in the period for each ticker.

This notebook compares the breakout + ADX accumulation strategy to monthly DCA for each ticker, focusing on **Final Amount / Total Invested** (Return Multiple).


In [75]:
# Imports and Configuration

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import talib
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Configuration
tickers = ['QQQ', 'TSLA', 'MSFT', 'GOOG', 'AAPL']
start_date = '2011-01-01'
end_date = '2025-12-31'

print('Tickers:', tickers)
print('Period: {} to {}'.format(start_date, end_date))


Tickers: ['QQQ', 'TSLA', 'MSFT', 'GOOG', 'AAPL']
Period: 2011-01-01 to 2025-12-31


In [76]:
# Data Download for Multiple Tickers

print('Downloading data for tickers...')
data_raw = {}

buffer_start = '2010-01-01'  # buffer for SMA200 & ADX

for ticker in tickers:
    print('
Downloading {} data...'.format(ticker))
    df = yf.download(ticker, start=buffer_start, end=end_date, progress=False, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    required = ['Open', 'High', 'Low', 'Close', 'Volume']
    if 'Adj Close' in df.columns:
        required.append('Adj Close')
    df = df[required].copy()
    if 'Adj Close' not in df.columns:
        df['Adj Close'] = df['Close']
    data_raw[ticker] = df
    print('  {}: {} records, {} to {}'.format(ticker, len(df), df.index[0], df.index[-1]))

print('
Downloaded data for {} tickers.'.format(len(data_raw)))


SyntaxError: EOL while scanning string literal (3795576951.py, line 9)

In [77]:
# Calculate SMA200, ADX, and Breakout + ADX Signals (Per Ticker)

max_entries_per_year = 12

data_sig = {}

for ticker in tickers:
    df = data_raw[ticker].copy()
    # Filter to analysis period
    df = df[(df.index >= start_date) & (df.index <= end_date)]

    # SMA200
    df['SMA200'] = df['Close'].rolling(window=200, min_periods=200).mean()

    # ADX14
    df['ADX14'] = talib.ADX(df['High'].values, df['Low'].values, df['Close'].values, timeperiod=14)

    # Breakout: Close crosses above SMA200 from below/at
    df['Breakout'] = (df['Close'] > df['SMA200']) & (df['Close'].shift(1) <= df['SMA200'].shift(1))

    # Next-day entry with ADX filter and Open > SMA200
    df['NextDay_Entry'] = False
    breakout_yday = df['Breakout'].shift(1) == True
    adx_strong = df['ADX14'].shift(1) > 25
    open_above_sma = df['Open'] > df['SMA200']
    df.loc[breakout_yday & adx_strong & open_above_sma, 'NextDay_Entry'] = True

    # Drop NaNs where SMA/ADX not available
    df = df.dropna(subset=['SMA200', 'ADX14'])

    # Enforce 12 buys per year cap
    df['Year'] = df.index.year
    yearly_counts = {y: 0 for y in df['Year'].unique()}
    entry_flags = []
    for dt, row in df.iterrows():
        if row['NextDay_Entry']:
            y = row['Year']
            if yearly_counts[y] < max_entries_per_year:
                entry_flags.append(True)
                yearly_counts[y] += 1
            else:
                entry_flags.append(False)
        else:
            entry_flags.append(False)
    df['Entry_Signal'] = entry_flags

    data_sig[ticker] = df

    print('
{} - Signals'.format(ticker))
    print('  Total Breakouts:', int(df['Breakout'].sum()))
    print('  Total NextDay_Entry (pre-cap):', int((breakout_yday & adx_strong & open_above_sma).sum()))
    print('  Total Entry_Signal (post-cap):', int(df['Entry_Signal'].sum()))
    print('  Entries per year (post-cap):')
    for y, grp in df[df['Entry_Signal']].groupby('Year'):
        print('    {}: {} entries'.format(y, len(grp)))

print('
Signals prepared for all tickers.')


SyntaxError: EOL while scanning string literal (3437742002.py, line 49)

In [78]:
# Simulate Breakout + ADX Accumulation and Monthly DCA Per Ticker

invest_amount_per_signal = 10000.0  # $10k per breakout entry
dca_invest_per_month = 10000.0      # $10k per month for DCA
max_total_dca_invested = 120000.0   # DCA cap (12 buys total)

results = []

for ticker in tickers:
    print('\n' + '='*60)
    print('Ticker: {}'.format(ticker))
    print('='*60)
    df = data_sig[ticker].copy()

    # --- Breakout + ADX Accumulation ---
    total_invested = 0.0
    positions = []  # (date, entry_price, shares)
    entry_dates = []

    for dt, row in df.iterrows():
        if row['Entry_Signal'] and total_invested < max_total_dca_invested:
            invest = min(invest_amount_per_signal, max_total_dca_invested - total_invested)
            if invest > 0:
                entry_price = row['Open']
                shares = invest / entry_price
                positions.append((dt, entry_price, shares))
                total_invested += invest
                entry_dates.append(dt)
                print('  Entry {}: {} @ ${:.2f}, Shares: {:.2f}, Invested: ${:,.0f}'.format(
                    len(positions), dt.strftime('%Y-%m-%d'), entry_price, shares, invest))

    final_close = df['Adj Close'].iloc[-1]
    final_signal_value = sum(shares * final_close for _, _, shares in positions)
    signal_ratio = final_signal_value / total_invested if total_invested > 0 else 0.0
    avg_signal_entry = np.mean([p for _, p, _ in positions]) if positions else 0.0

    print('  Signal Entries: {}, Invested: ${:,.0f}, Final: ${:,.0f}, Ratio: {:.2f}x'.format(
        len(positions), total_invested, final_signal_value, signal_ratio))

    # --- Monthly DCA Benchmark ---
    total_dca_invested = 0.0
    dca_shares = 0.0
    dca_entry_prices = []
    dca_investment_count = 0

    monthly = df.resample('ME').last()
    monthly = monthly[(monthly.index >= start_date) & (monthly.index <= end_date)]

    for dt, row in monthly.iterrows():
        if total_dca_invested >= max_total_dca_invested:
            break
        invest = min(dca_invest_per_month, max_total_dca_invested - total_dca_invested)
        price = row['Adj Close']
        shares = invest / price
        dca_shares += shares
        total_dca_invested += invest
        dca_investment_count += 1
        dca_entry_prices.append(price)
        print('  DCA Entry {}: {} @ ${:.2f}, Shares: {:.2f}, Invested: ${:,.0f}'.format(
            dca_investment_count, dt.strftime('%Y-%m-%d'), price, shares, invest))

    final_dca_value = dca_shares * final_close
    dca_ratio = final_dca_value / total_dca_invested if total_dca_invested > 0 else 0.0
    avg_dca_entry = np.mean(dca_entry_prices) if dca_entry_prices else 0.0

    print('  DCA Buys: {}, Invested: ${:,.0f}, Final: ${:,.0f}, Ratio: {:.2f}x'.format(
        dca_investment_count, total_dca_invested, final_dca_value, dca_ratio))

    results.append({
        'Ticker': ticker,
        'Signal Buys': len(positions),
        'Signal Invested': total_invested,
        'Signal Final': final_signal_value,
        'Signal Ratio': signal_ratio,
        'Signal Avg Entry': avg_signal_entry,
        'DCA Buys': dca_investment_count,
        'DCA Invested': total_dca_invested,
        'DCA Final': final_dca_value,
        'DCA Ratio': dca_ratio,
        'DCA Avg Entry': avg_dca_entry,
        'Final Close': final_close,
    })

print('\nSimulation complete for all tickers.')



Ticker: QQQ


NameError: name 'data_sig' is not defined

In [79]:
# Summary Comparison Table Across All Tickers

results_df = pd.DataFrame(results)

cols_to_show = [
    'Ticker',
    'Signal Buys', 'Signal Invested', 'Signal Final', 'Signal Ratio',
    'DCA Buys', 'DCA Invested', 'DCA Final', 'DCA Ratio'
]

print('\n' + '='*60)
print('MULTI-TICKER COMPARISON - BREAKOUT + ADX vs DCA (2011-2025)')
print('='*60)
print(results_df[cols_to_show].to_string(index=False))
print('='*60)



MULTI-TICKER COMPARISON - BREAKOUT + ADX vs DCA (2011-2025)


KeyError: "None of [Index(['Ticker', 'Signal Buys', 'Signal Invested', 'Signal Final',\n       'Signal Ratio', 'DCA Buys', 'DCA Invested', 'DCA Final', 'DCA Ratio'],\n      dtype='object')] are in the [columns]"

## Conclusion

**Breakout + ADX Accumulation Strategy (per ticker):**
- Buys only when 200 SMA breakout occurs in a strong trend (ADX(14) > 25).
- Next-day confirmation at Open > SMA200; up to 12 buys per year, $10k each.
- No stop loss or exits; all positions held to the end of 2025.

**Monthly DCA Benchmark (per ticker):**
- Buys $10,000 at month-end Close each month until $120,000 total is reached (12 buys).
- Ignores regime; buys through bull, bear, and sideways markets.

**Comparison Metric:**
- **Return Multiple** = Final Portfolio Value / Total Invested for each method and ticker.
- Higher multiple indicates better final wealth per dollar invested.

Use the summary table to see, for each ticker (QQQ, TSLA, MSFT, GOOG, AAPL), whether the selective breakout + ADX accumulation approach improved capital efficiency versus simple monthly DCA over 2011–2025.
